# Artificial Intelligence — Lab 8
## CSP Heuristics and Forward Checking

**Course Learning Outcome — CLO4**  
Investigate Constraint Satisfaction Problems (CSPs), their representations, constraints, heuristics, and propagation methods.

**Environment:** Python 3 / Jupyter Notebook  
**Submission:** completed notebook containing predictions, traces, code, experiments, justifications, debugging answers, and reflection.

> **Assessment principle:** Correct code is only one part of the evidence. Most marks come from your ability to **justify heuristic choices, predict domain reductions, explain why forward checking detects failure earlier, and interpret how MRV, degree heuristic, and LCV affect search effort**.

## Lab at a Glance

| Stage | Suggested time | What you will do |
|---|---:|---|
| 1. Heuristic reasoning | 15 min | Predict MRV, degree, and LCV choices |
| 2. Implement variable/value ordering | 30 min | Add MRV, degree heuristic, and LCV |
| 3. Implement forward checking | 30 min | Propagate domain reductions after assignments |
| 4. Compare solver variants | 25 min | Measure assignments, failures, and backtracks |
| 5. Diagnose propagation failures | 10 min | Explain domain wipeout and restoration |
| 6. Personalized variation & reflection | 10 min | Defend conclusions on a modified CSP |

> **Main idea:** CSP heuristics do not change the set of valid solutions. They change **which parts of the search space are explored first** and how early failure can be detected.

## Learning Objectives

By the end of this lab, you should be able to:

1. explain the **Minimum Remaining Values (MRV)** heuristic;
2. explain the **degree heuristic** as a tie-breaker;
3. explain the **Least-Constraining Value (LCV)** heuristic;
4. predict which variable/value a heuristic will select;
5. implement MRV, degree, and LCV;
6. explain and implement **forward checking**;
7. identify a **domain wipeout**;
8. restore domains correctly during backtracking;
9. compare plain backtracking with heuristic/propagation-enhanced variants;
10. justify performance differences using empirical evidence.

In [ ]:
from copy import deepcopy
from typing import Dict, List, Set, Tuple, Optional

print("Lab 8 environment ready.")

# Part I — Recall the CSP

We again use the Australia map-coloring CSP.

Variables:

```text
WA, NT, SA, Q, NSW, V, T
```

Colors:

```text
Red, Green, Blue
```

Neighboring regions must have different colors.

In [ ]:
VARIABLES = ["WA", "NT", "SA", "Q", "NSW", "V", "T"]
COLORS = ["Red", "Green", "Blue"]

DOMAINS = {
    var: list(COLORS)
    for var in VARIABLES
}

NEIGHBORS = {
    "WA": {"NT", "SA"},
    "NT": {"WA", "SA", "Q"},
    "SA": {"WA", "NT", "Q", "NSW", "V"},
    "Q": {"NT", "SA", "NSW"},
    "NSW": {"SA", "Q", "V"},
    "V": {"SA", "NSW"},
    "T": set(),
}

In [ ]:
def is_consistent(variable, value, assignment, neighbors):
    for neighbor in neighbors[variable]:
        if neighbor in assignment and assignment[neighbor] == value:
            return False
    return True

# Part II — Minimum Remaining Values (MRV)

MRV chooses the unassigned variable with the **smallest current domain**:

$$
X_{\text{MRV}}
=
\arg\min_{X_i \text{ unassigned}}
|D_i|.
$$

It is often called the **most constrained variable** heuristic.

The intuition is:

> Try the variable that is closest to failure first.

If that variable has no legal value, we discover the failure early rather than deep in the search tree.

## Task 2.1 — Predict MRV

Suppose the current domains are:

```text
WA  -> {Red, Green}
NT  -> {Green}
SA  -> {Red, Blue}
Q   -> {Red, Green, Blue}
NSW -> {Red, Blue}
V   -> {Green, Blue}
T   -> {Red, Green, Blue}
```

and nothing is assigned yet.

1. Which variable should MRV select?
2. Why?
3. If two variables both have two remaining values, does MRV alone determine which one to choose?
4. What kind of additional heuristic can break such a tie?

**Your answers:**

## Task 2.2 — Implement MRV

Complete the function below.

It should consider **only unassigned variables** and return one having the smallest current domain.

In [ ]:
def select_mrv_variable(variables, domains, assignment):
    # TODO:
    # 1. collect unassigned variables
    # 2. return the variable whose current domain is smallest
    raise NotImplementedError

### Self-check

Predict the result before running.

In [ ]:
test_domains = deepcopy(DOMAINS)
test_domains["WA"] = ["Red", "Green"]
test_domains["NT"] = ["Green"]
test_domains["SA"] = ["Red", "Blue"]

predicted = "NT"
print("Selected:", select_mrv_variable(VARIABLES, test_domains, {}))
assert select_mrv_variable(VARIABLES, test_domains, {}) == predicted
print("MRV test passed.")

# Part III — Degree Heuristic

When MRV produces a tie, the **degree heuristic** prefers the variable that constrains the largest number of other **unassigned** variables.

One possible formulation is:

$$
\deg(X_i)
=
|\{X_j \in N(X_i): X_j \text{ is unassigned}\}|.
$$

We choose the tied variable with the **largest** degree.

## Task 3.1 — Predict the Degree Heuristic

At the beginning of the Australia CSP, all domains have size 3.

1. Which variable has the highest degree?
2. How many neighbors does it constrain?
3. Why might selecting this variable early reduce later branching?

**Your answers:**

## Task 3.2 — Implement MRV + Degree Tie-Breaking

Complete the function.

Rules:

1. find the minimum remaining domain size;
2. keep only variables tied for MRV;
3. among them, choose the variable with the greatest number of **unassigned neighbors**.

In [ ]:
def select_mrv_degree_variable(variables, domains, assignment, neighbors):
    # TODO:
    # Implement MRV first, then degree heuristic for ties.
    raise NotImplementedError

### Self-check

At the start, all domains are equal. The degree heuristic should prefer `SA`.

In [ ]:
selected = select_mrv_degree_variable(
    VARIABLES, DOMAINS, {}, NEIGHBORS
)

print("Selected:", selected)
assert selected == "SA"
print("MRV + degree test passed.")

## Task 3.3 — Explain the Difference

Why are MRV and degree heuristic not the same idea?

Complete:

- MRV asks: **Which variable has the fewest ________?**
- Degree asks: **Which variable constrains the most ________?**

Then explain why using them together can be useful.

**Your answer:**

# Part IV — Least-Constraining Value (LCV)

LCV chooses the value that removes the **fewest options** from neighboring unassigned variables.

For candidate value $v$ of variable $X$:

$$
\operatorname{impact}(v)
=
\text{number of neighbor-domain values eliminated by assigning } X=v.
$$

LCV prefers the candidate with the smallest impact.

The intuition is:

> Leave as much flexibility as possible for the remaining variables.

## Task 4.1 — Manual LCV Prediction

Suppose:

```text
SA domain = {Red, Green, Blue}
WA domain = {Red, Green}
NT domain = {Red, Blue}
Q  domain = {Green, Blue}
NSW domain= {Red, Green}
V domain  = {Blue}
```

All these variables are currently unassigned.

For each possible value of `SA`, count how many values would be removed from its neighbors.

| Candidate for SA | Values eliminated from neighbors | Total impact |
|---|---|---:|
| Red |  |  |
| Green |  |  |
| Blue |  |  |

Then answer:

1. Which color is least constraining?
2. If two colors have equal impact, what could the solver do?
3. Does LCV guarantee that the chosen value belongs to a global solution?

**Your answers:**

## Task 4.2 — Implement LCV

Complete the function below.

Return the values in the variable's current domain ordered from **least constraining** to **most constraining**.

In [ ]:
def order_lcv_values(variable, domains, assignment, neighbors):
    # TODO:
    # For each candidate value:
    #   count how many occurrences of that value appear
    #   in the domains of unassigned neighbors.
    #
    # Return the domain values sorted by increasing impact.
    raise NotImplementedError

### LCV self-check

In [ ]:
lcv_domains = deepcopy(DOMAINS)
lcv_domains["SA"] = ["Red", "Green", "Blue"]
lcv_domains["WA"] = ["Red", "Green"]
lcv_domains["NT"] = ["Red", "Blue"]
lcv_domains["Q"] = ["Green", "Blue"]
lcv_domains["NSW"] = ["Red", "Green"]
lcv_domains["V"] = ["Blue"]

ordered = order_lcv_values("SA", lcv_domains, {}, NEIGHBORS)
print("LCV order for SA:", ordered)

## Task 4.3 — Explain LCV

1. Why is LCV called a **value-ordering** heuristic rather than a variable-ordering heuristic?
2. Why does LCV try to preserve flexibility?
3. Why can a least-constraining value still eventually lead to failure?

**Your answers:**

# Part V — Forward Checking

Forward checking performs limited constraint propagation after assigning a variable.

If we assign

```text
SA = Red
```

then `Red` must be removed from the domains of every unassigned neighbor of `SA`.

If any unassigned variable's domain becomes empty, we have a **domain wipeout** and the current branch must fail immediately.

## Task 5.1 — Predict Domain Reduction

Suppose every variable initially has domain:

```text
{Red, Green, Blue}
```

and we assign:

```text
SA = Red
```

Predict the resulting domains of:

- `WA`
- `NT`
- `Q`
- `NSW`
- `V`
- `T`

Then answer:

> Why is the domain of `T` unchanged?

**Your prediction:**

## Task 5.2 — Implement Forward Checking

The function should:

1. receive the newly assigned variable/value;
2. remove the assigned value from every unassigned neighbor's domain;
3. return `False` immediately if any domain becomes empty;
4. otherwise return `True`.

The function should modify the supplied domain dictionary.

In [ ]:
def forward_check(variable, value, domains, assignment, neighbors):
    # TODO:
    # Remove 'value' from domains of each unassigned neighbor.
    # Return False if any domain becomes empty.
    # Otherwise return True.
    raise NotImplementedError

### Forward-checking self-check

Before running, predict the domain of `WA` after `SA=Red`.

In [ ]:
fc_domains = deepcopy(DOMAINS)
fc_assignment = {"SA": "Red"}

ok = forward_check(
    "SA", "Red",
    fc_domains,
    fc_assignment,
    NEIGHBORS
)

print("Forward checking successful:", ok)
print("WA domain:", fc_domains["WA"])
print("NT domain:", fc_domains["NT"])
print("Q domain:", fc_domains["Q"])
print("NSW domain:", fc_domains["NSW"])
print("V domain:", fc_domains["V"])
print("T domain:", fc_domains["T"])

assert "Red" not in fc_domains["WA"]
assert "Red" not in fc_domains["NT"]
assert fc_domains["T"] == COLORS
print("Forward-checking test passed.")

## Task 5.3 — Domain Wipeout

Suppose:

```text
WA domain = {Red}
SA is assigned Red
```

What happens during forward checking?

1. What becomes the domain of `WA`?
2. Why must the search branch fail immediately?
3. Why is discovering this now better than discovering it several recursive levels later?

**Your answer:**

# Part VI — Why Domain Restoration Is Necessary

During backtracking, domains modified by a failed branch must not incorrectly remain reduced when another branch is explored.

A simple safe approach for this lab is:

> Before trying a value, make a deep copy of the current domains.

This is not the most memory-efficient technique, but it makes the backtracking logic easier to reason about.

## Task 6.1 — Reason About Restoration

Suppose:

1. you try `SA=Red`;
2. forward checking removes `Red` from several neighboring domains;
3. the branch later fails;
4. you next try `SA=Green`.

Why would it be wrong to keep all domain reductions caused specifically by `SA=Red`?

**Your answer:**

# Part VII — Build an Enhanced CSP Solver

We will create a solver that can switch the following options on/off:

- MRV;
- degree tie-breaking;
- LCV;
- forward checking.

This allows us to compare solver behavior experimentally.

In [ ]:
def select_variable(
    variables,
    domains,
    assignment,
    neighbors,
    use_mrv=False,
    use_degree=False,
):
    if not use_mrv:
        for var in variables:
            if var not in assignment:
                return var

    if use_mrv and use_degree:
        return select_mrv_degree_variable(
            variables, domains, assignment, neighbors
        )

    return select_mrv_variable(
        variables, domains, assignment
    )


def order_values(
    variable,
    domains,
    assignment,
    neighbors,
    use_lcv=False,
):
    if use_lcv:
        return order_lcv_values(
            variable, domains, assignment, neighbors
        )
    return list(domains[variable])

In [ ]:
def solve_csp(
    variables,
    initial_domains,
    neighbors,
    *,
    use_mrv=False,
    use_degree=False,
    use_lcv=False,
    use_forward_checking=False,
):
    stats = {
        "assignments_tried": 0,
        "consistency_failures": 0,
        "domain_wipeouts": 0,
        "backtracks": 0,
        "max_depth": 0,
        "trace": [],
    }

    def backtrack(assignment, domains):
        stats["max_depth"] = max(
            stats["max_depth"], len(assignment)
        )

        # Complete assignment
        if len(assignment) == len(variables):
            return dict(assignment)

        variable = select_variable(
            variables,
            domains,
            assignment,
            neighbors,
            use_mrv=use_mrv,
            use_degree=use_degree,
        )

        values = order_values(
            variable,
            domains,
            assignment,
            neighbors,
            use_lcv=use_lcv,
        )

        for value in values:
            stats["assignments_tried"] += 1

            consistent = is_consistent(
                variable, value, assignment, neighbors
            )

            stats["trace"].append({
                "depth": len(assignment),
                "variable": variable,
                "value": value,
                "consistent": consistent,
            })

            if not consistent:
                stats["consistency_failures"] += 1
                continue

            assignment[variable] = value

            # Make a separate domain state for this branch.
            next_domains = deepcopy(domains)
            next_domains[variable] = [value]

            propagation_ok = True

            if use_forward_checking:
                # TODO:
                # call forward_check(...)
                # store result in propagation_ok
                pass

            if not propagation_ok:
                stats["domain_wipeouts"] += 1
                del assignment[variable]
                continue

            result = backtrack(
                assignment,
                next_domains
            )

            if result is not None:
                return result

            del assignment[variable]

        stats["backtracks"] += 1
        return None

    solution = backtrack(
        {},
        deepcopy(initial_domains)
    )

    return solution, stats

## Task 7.1 — Complete the Solver

Finish the `use_forward_checking` section above.

Then explain:

1. Why is `next_domains` a copy?
2. Why do we set `next_domains[variable] = [value]`?
3. Why do we delete the assignment after a recursive failure?
4. What does `domain_wipeouts` measure?

**Your answers:**

# Part VIII — Compare Solver Variants

We compare four strategies:

1. **Plain backtracking**
2. **MRV + degree**
3. **MRV + degree + LCV**
4. **MRV + degree + LCV + forward checking**

Before running, predict which one will generally perform the least unnecessary search.

## Task 8.1 — Prediction

Rank the four variants from **least sophisticated** to **most informed**.

Then predict:

- which should try the fewest assignments;
- which should detect impossible branches earliest;
- whether all should still find a valid solution.

**Your prediction:**

In [ ]:
configurations = [
    {
        "name": "Plain backtracking",
        "use_mrv": False,
        "use_degree": False,
        "use_lcv": False,
        "use_forward_checking": False,
    },
    {
        "name": "MRV + degree",
        "use_mrv": True,
        "use_degree": True,
        "use_lcv": False,
        "use_forward_checking": False,
    },
    {
        "name": "MRV + degree + LCV",
        "use_mrv": True,
        "use_degree": True,
        "use_lcv": True,
        "use_forward_checking": False,
    },
    {
        "name": "MRV + degree + LCV + FC",
        "use_mrv": True,
        "use_degree": True,
        "use_lcv": True,
        "use_forward_checking": True,
    },
]

results = []

for cfg in configurations:
    solution, stats = solve_csp(
        VARIABLES,
        DOMAINS,
        NEIGHBORS,
        use_mrv=cfg["use_mrv"],
        use_degree=cfg["use_degree"],
        use_lcv=cfg["use_lcv"],
        use_forward_checking=cfg["use_forward_checking"],
    )

    results.append({
        "name": cfg["name"],
        "solution": solution,
        **stats,
    })

for row in results:
    print("\n" + row["name"])
    print(" solution:", row["solution"])
    print(" assignments tried:", row["assignments_tried"])
    print(" failures:", row["consistency_failures"])
    print(" wipeouts:", row["domain_wipeouts"])
    print(" backtracks:", row["backtracks"])

## Task 8.2 — Analyze the Comparison

Complete:

| Solver | Assignments tried | Consistency failures | Domain wipeouts | Backtracks |
|---|---:|---:|---:|---:|
| Plain |  |  |  |  |
| MRV + degree |  |  |  |  |
| + LCV |  |  |  |  |
| + Forward checking |  |  |  |  |

Then answer:

1. Did all variants find a valid solution?
2. Which variant performed the least search on this instance?
3. Did every additional heuristic reduce every metric?
4. Why can small/easy CSPs show only a small difference between strategies?
5. Why should CSP heuristics be evaluated on multiple problem instances rather than one tiny example?

**Your answers:**

# Part IX — A More Constrained Scheduling CSP

We now test the same solver on a scheduling problem.

Variables:

```text
AI, DB, Networks, Security, Programming, Math
```

Slots:

```text
S1, S2, S3
```

Conflicts mean the two courses cannot share a slot.

In [ ]:
COURSES = [
    "AI",
    "DB",
    "Networks",
    "Security",
    "Programming",
    "Math",
]

SLOTS = ["S1", "S2", "S3"]

SCHEDULE_DOMAINS = {
    course: list(SLOTS)
    for course in COURSES
}

SCHEDULE_NEIGHBORS = {
    "AI": {"DB", "Networks", "Security"},
    "DB": {"AI", "Networks", "Programming"},
    "Networks": {"AI", "DB", "Security", "Math"},
    "Security": {"AI", "Networks", "Programming"},
    "Programming": {"DB", "Security", "Math"},
    "Math": {"Networks", "Programming"},
}

## Task 9.1 — Predict the First Variable

At the beginning, all variables have domains of equal size.

If MRV is tied and the degree heuristic is used:

1. Which course should be selected first?
2. Why?
3. How many unassigned neighbors does it constrain?

**Your prediction:**

In [ ]:
plain_solution, plain_stats = solve_csp(
    COURSES,
    SCHEDULE_DOMAINS,
    SCHEDULE_NEIGHBORS,
)

smart_solution, smart_stats = solve_csp(
    COURSES,
    SCHEDULE_DOMAINS,
    SCHEDULE_NEIGHBORS,
    use_mrv=True,
    use_degree=True,
    use_lcv=True,
    use_forward_checking=True,
)

print("PLAIN")
print(" solution:", plain_solution)
print(" assignments:", plain_stats["assignments_tried"])
print(" failures:", plain_stats["consistency_failures"])
print(" backtracks:", plain_stats["backtracks"])

print("\nHEURISTICS + FC")
print(" solution:", smart_solution)
print(" assignments:", smart_stats["assignments_tried"])
print(" failures:", smart_stats["consistency_failures"])
print(" wipeouts:", smart_stats["domain_wipeouts"])
print(" backtracks:", smart_stats["backtracks"])

## Task 9.2 — Interpret the Scheduling Experiment

1. Which solver tried fewer assignments?
2. Did both find a valid schedule?
3. Why is degree heuristic especially natural for this conflict graph?
4. Why can forward checking discover an impossible branch before assigning every course?
5. Which metric most directly counts how many candidate assignments the algorithm attempted?

**Your answers:**

# Part X — Debugging CSP Heuristics and Propagation

## Task 10.1 — Faulty MRV

A student writes:

```python
return max(unassigned, key=lambda v: len(domains[v]))
```

1. What does this choose?
2. Why is it the opposite of MRV?
3. What should replace `max`?

**Your answer:**

## Task 10.2 — Faulty Degree Heuristic

A student counts **all** neighbors, including already-assigned variables.

1. Why is this less informative than counting only unassigned neighbors?
2. Which neighbors still create future constraints?

**Your answer:**

## Task 10.3 — Faulty Forward Checking

A student removes a value from a neighbor's domain but never restores the domain after backtracking.

1. Why can this remove legal values from unrelated later branches?
2. What incorrect outcome could result?
3. Give one safe restoration strategy.

**Your answer:**

## Task 10.4 — Missing Domain-Wipeout Check

A student allows an unassigned variable's domain to become empty but continues recursion.

1. Why is the branch already impossible?
2. Why should the algorithm return failure immediately?
3. How does this illustrate the purpose of constraint propagation?

**Your answer:**

# Part XI — Personalized CSP Variation

Use the last digit of your student ID.

- `0–3`: restrict `AI` to `{S1, S2}`
- `4–6`: restrict `Networks` to `{S2, S3}`
- `7–9`: restrict `Programming` to `{S1, S3}`

Create a copy of `SCHEDULE_DOMAINS` and apply only your assigned restriction.

In [ ]:
LAST_DIGIT = None  # TODO: replace with an integer from 0 to 9

personal_domains = deepcopy(SCHEDULE_DOMAINS)

if LAST_DIGIT is not None:
    if 0 <= LAST_DIGIT <= 3:
        personal_domains["AI"] = ["S1", "S2"]
        PERSONAL_CHANGE = "AI -> {S1, S2}"
    elif 4 <= LAST_DIGIT <= 6:
        personal_domains["Networks"] = ["S2", "S3"]
        PERSONAL_CHANGE = "Networks -> {S2, S3}"
    elif 7 <= LAST_DIGIT <= 9:
        personal_domains["Programming"] = ["S1", "S3"]
        PERSONAL_CHANGE = "Programming -> {S1, S3}"
    else:
        raise ValueError("LAST_DIGIT must be between 0 and 9")

    print("Assigned variation:", PERSONAL_CHANGE)

## Task 11.1 — Predict Before Running

Write:

- **My domain restriction:**  
- **Which CSP component changed: $X$, $D$, or $C$?**  
- **Which variable do I expect MRV to select first after this change?**  
- **Why?**  
- **Do I expect forward checking to become more or less important?**

**Your prediction:**

In [ ]:
if LAST_DIGIT is not None:
    personal_solution, personal_stats = solve_csp(
        COURSES,
        personal_domains,
        SCHEDULE_NEIGHBORS,
        use_mrv=True,
        use_degree=True,
        use_lcv=True,
        use_forward_checking=True,
    )

    print("Personal solution:", personal_solution)
    print("Assignments tried:", personal_stats["assignments_tried"])
    print("Consistency failures:", personal_stats["consistency_failures"])
    print("Domain wipeouts:", personal_stats["domain_wipeouts"])
    print("Backtracks:", personal_stats["backtracks"])

## Task 11.2 — Explain the Personalized Result

1. Was your MRV prediction correct?
2. Did the CSP remain solvable?
3. Did the domain restriction reduce or increase search effort?
4. Why can restricting a domain sometimes help the search?
5. Why can the same kind of restriction also make another CSP unsatisfiable?

**Your answers:**

# Part XII — Individual Understanding Check

Your instructor may ask one short question about your notebook.

Possible prompts:

- Show me where MRV selects a variable.
- Why is `SA` favored by the degree heuristic?
- What does LCV try to preserve?
- Show me where forward checking removes a value.
- What is a domain wipeout?
- Why do we copy domains before recursion?
- Why can heuristics change runtime without changing the valid solutions?
- Which component changed in your personalized variation?

> You are expected to explain the **AI concept represented by the code**, not memorize Python syntax.

# Reflection

Answer concisely but precisely.

### R1 — MRV
Why is choosing the most constrained variable often called a “fail-first” strategy?

**Answer:**

### R2 — Degree Heuristic
Why is degree heuristic useful mainly as a tie-breaker for MRV?

**Answer:**

### R3 — LCV
Why does LCV prefer a value that removes fewer options from neighboring variables?

**Answer:**

### R4 — Forward Checking
What kind of inconsistency can forward checking detect earlier than plain backtracking?

**Answer:**

### R5 — Heuristics vs. Problem Definition
Do MRV, degree heuristic, and LCV change the CSP's valid solutions? Explain.

**Answer:**

# Submission Checklist

Before submitting, verify that your notebook contains:

- [ ] MRV prediction and implementation;
- [ ] degree-heuristic reasoning and implementation;
- [ ] manual LCV calculation;
- [ ] working LCV ordering;
- [ ] forward-checking prediction and implementation;
- [ ] domain-wipeout explanation;
- [ ] domain-restoration justification;
- [ ] completed enhanced solver;
- [ ] comparison of four solver variants;
- [ ] scheduling-CSP experiment;
- [ ] debugging answers;
- [ ] personalized domain variation;
- [ ] prediction before personalized execution;
- [ ] reflection answers;
- [ ] visible outputs from important code cells.

Suggested filename:

```text
Lab08_StudentID.ipynb
```

# Assessment Guide — 10 Marks

| Component | Marks | Evidence expected |
|---|---:|---|
| **Correct implementation** | **2.0** | MRV/degree, LCV, forward checking, and enhanced solver work correctly |
| **Algorithmic / modeling justification** | **3.0** | Explains heuristic choices, domain reductions, propagation, wipeout, and restoration |
| **Experimental analysis** | **2.0** | Interprets solver comparisons and scheduling-CSP performance |
| **Trace / prediction / debugging** | **1.0** | Manual heuristic predictions, domain reasoning, and faulty-code diagnosis |
| **Individual understanding check** | **1.0** | Short explanation of selected part of the student's own work |
| **Code quality & completeness** | **1.0** | Readable code, complete responses, required outputs |
| **Total** | **10.0** |  |

> **Key rule:** Correct code without adequate explanation earns only a limited portion of the marks.

## Key Takeaways

- **MRV** selects the variable with the fewest remaining values.
- The **degree heuristic** prefers a variable that constrains many unassigned neighbors.
- **LCV** chooses the value that removes the fewest options from neighboring variables.
- **Forward checking** propagates an assignment by reducing neighbor domains.
- An empty domain indicates a **domain wipeout** and immediate failure of the current branch.
- Domains must be correctly restored when backtracking.
- Heuristics and propagation do not change which assignments are valid; they change **how efficiently the solver searches**.

The next lab will move to **Adversarial Search and Minimax Game Playing**.